In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import apache_beam as beam
from apache_beam.io.parquetio import ReadFromParquet, WriteToParquet
import csv

Failed to import GCSFileSystem; loading of this filesystem will be skipped. Error details: cannot import name 'storage' from 'google.cloud' (unknown location)


In [ ]:
data_inputs = "../data/inputs/"
data_outputs_psa = "../data/outputs/psa/"
data_outputs_errors = "../data/outputs/errors/"
data_outputs_gold = "../data/outputs/gold/"

In [4]:
def get_proc_date():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")

proc_date = get_proc_date()
print(proc_date)

2026-06-29


In [5]:
def read_psa_path(proc_date):
    return str(
        Path(data_outputs_psa)/ f"proc_date={proc_date}"/ "sales*.parquet")

read_psa_path(proc_date=proc_date)


'..\\data\\outputs\\psa\\proc_date=2026-06-29\\sales*.parquet'

In [ ]:
class ParsearYValidarContrato(beam.DoFn):
    def process(self, row):
        errores = []

        try:
            monto_original = float(row["monto_original"])

            if monto_original <= 0:
                errores.append("monto menor a cero")

        except Exception:
            errores.append("Monto no corresponde a un formato numerico valido (float/int)")
            monto_original = None

        ciudad = str(row.get("ciudad", ""))

        if ciudad not in ["Santiago", "Lima", "Buenos Aires"]:
            errores.append("Ciudad no autorizada en contrato regional")

        if errores:
            yield beam.pvalue.TaggedOutput(
                "corruptos",
                {
                    "raw_record": dict(row),
                    "motivo_rechazo": " o ".join(errores),
                },
            )
            return

        yield {
            "id_transaccion": str(row["id_transaccion"]),
            "ciudad": ciudad,
            "monto_original": monto_original,
            "moneda_origen": str(row["moneda_origen"]),
            "fecha_transaccion": str(row["fecha_transaccion"]),
        }

In [ ]:
def cargar_tipo_cambio():
    tipo_cambio = {}

    with open(Path(data_inputs) / "tipo_cambio.csv", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            tipo_cambio[row["codigo_moneda"]] = float(row["factor_usd"])

    return tipo_cambio


TIPO_CAMBIO = cargar_tipo_cambio()

In [ ]:
class ConvertirGoldUSD(beam.DoFn):
    def process(self, row):
        factor_usd = TIPO_CAMBIO[row["moneda_origen"]]

        yield {
            "id_transaccion": row["id_transaccion"],
            "ciudad": row["ciudad"],
            "monto_usd": round(row["monto_original"] * factor_usd, 3),
            "fecha_compras": row["fecha_transaccion"],
        }

In [11]:
import pyarrow as pa

schema_gold = pa.schema([
    ("id_transaccion", pa.string()),
    ("ciudad", pa.string()),
    ("monto_usd", pa.float64()),
    ("fecha_compras", pa.string()),
])

In [ ]:
proc_date = get_proc_date()
output_gold_path = (Path(data_outputs_gold)/ f"proc_date={proc_date}"/ "sales")
output_errors_path = (Path(data_outputs_errors)/ f"proc_date={proc_date}"/ "sales")

In [ ]:
with beam.Pipeline() as p:

    resultado = (
        p
        | "Leer PSA" >> ReadFromParquet(read_psa_path(proc_date))
        | "Parsear y validar contrato" >> beam.ParDo(
            ParsearYValidarContrato()
        ).with_outputs(
            "corruptos",
            main="validos"
        )
    )

    validos = resultado.validos
    corruptos = resultado.corruptos

    gold = (
        validos
        | "Convertir monto a USD" >> beam.ParDo(ConvertirGoldUSD())
    )

    (
        corruptos
        | "Errores a JSON" >> beam.Map(
            lambda row: json.dumps(row, ensure_ascii=False)
        )
        | "Guardar DLQ" >> beam.io.WriteToText(
            file_path_prefix=str(output_errors_path),
            file_name_suffix=".json"
        )
    )

    (
        gold
        | "Guardar Capa Gold Parquet" >> WriteToParquet(
            file_path_prefix=str(output_gold_path),
            schema=schema_gold,
            file_name_suffix=".parquet"
        )
    )

ERROR:apache_beam.runners.common:"monto_usd [while running '[13]: Guardar Capa Gold Parquet/ParDo(_RowDictionariesToArrowTable)']"
Traceback (most recent call last):
  File "apache_beam\runners\common.py", line 1435, in apache_beam.runners.common.DoFnRunner.process
  File "apache_beam\runners\common.py", line 639, in apache_beam.runners.common.SimpleInvoker.invoke_process
  File "apache_beam\runners\common.py", line 1611, in apache_beam.runners.common._OutputHandler.handle_process_outputs
  File "c:\Users\Carlos\Desktop\data-dev\retail-elt-beam-airflow\.venv\Lib\site-packages\apache_beam\io\parquetio.py", line 119, in process
    self._buffer[i].append(row[n])
                           ~~~^^^
KeyError: 'monto_usd'


KeyError: "monto_usd [while running '[13]: Guardar Capa Gold Parquet/ParDo(_RowDictionariesToArrowTable)']"